# 03 — News context (hardened)
Best-effort news collection with graceful provider failure handling.

In [ ]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import yaml

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def safe_json_response(resp):
    try:
        return resp.json(), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

def env(name):
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / "configs").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

ROOT = find_repo_root(Path.cwd())
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
cfg = yaml.safe_load((CONFIGS / "run.yml").read_text(encoding="utf-8")) if (CONFIGS / "run.yml").exists() else {}
sites = cfg.get("sites", [])
run_cfg = cfg.get("run", {})
DATE_FROM = run_cfg.get("date_from", "2025-01-01")
DATE_TO = run_cfg.get("date_to", "2025-01-31")

def manifest_base(step):
    return {
        "step": step,
        "created_at_utc": utc_now(),
        "repo_root": str(ROOT),
        "configs": [{"path": str(CONFIGS / "run.yml"), "sha256": sha256_file(CONFIGS / "run.yml")}] if (CONFIGS / "run.yml").exists() else [],
        "artifacts": [],
        "notes": [],
        "status": "ok",
    }

def add_artifact(mf, p: Path, meta=None):
    if p.exists():
        row = {"path": str(p), "sha256": sha256_file(p)}
        if meta:
            row["meta"] = meta
        mf["artifacts"].append(row)

In [ ]:
step = "03_news_context"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

news_cfg = cfg.get("news") or {}
terms = news_cfg.get("query_terms")
if not terms:
    base_terms = ["Newhaven ERF","Newhaven incinerator","Lewes air quality","Brighton air quality","Eastbourne air quality","PM2.5 Sussex","NO2 Sussex"]
    seen = set()
    terms = []
    for s in sites:
        nm = (s.get("name") or "").strip()
        if nm and nm.lower() not in seen:
            seen.add(nm.lower()); terms.append(nm)
    for t in base_terms:
        if t.lower() not in seen:
            seen.add(t.lower()); terms.append(t)

articles = []
failures = []

def record_failure(provider, query, resp=None, reason=""):
    failures.append({
        "provider": provider,
        "query": query,
        "reason": reason,
        "status_code": getattr(resp, "status_code", None),
        "ok": getattr(resp, "ok", None),
        "content_type": (getattr(resp, "headers", {}) or {}).get("Content-Type", ""),
        "response_preview": (getattr(resp, "text", "") or "")[:500],
    })

newsapi_key = env("NEWSAPI_KEY") or env("NEWS_API_KEY")
newsdata_key = env("NEWS_DATA_IO_KEY")
gnews_key = env("GNEWS_API_KEY")

for term in terms:
    if newsapi_key:
        try:
            r = requests.get("https://newsapi.org/v2/everything",
                             params={"q":term,"pageSize":10,"sortBy":"publishedAt","language":"en"},
                             headers={"X-Api-Key": newsapi_key}, timeout=30)
            payload, err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("articles", []):
                    articles.append({"provider":"newsapi","query":term,"title":a.get("title"),"source":(a.get("source") or {}).get("name"),"publishedAt":a.get("publishedAt"),"url":a.get("url"),"description":a.get("description")})
            else:
                record_failure("newsapi", term, r, err or "non-ok response")
        except Exception as e:
            record_failure("newsapi", term, None, f"{type(e).__name__}: {e}")

    if gnews_key:
        try:
            r = requests.get("https://gnews.io/api/v4/search",
                             params={"q":term,"lang":"en","country":"gb","token":gnews_key,"max":10},
                             timeout=30)
            payload, err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("articles", []):
                    articles.append({"provider":"gnews","query":term,"title":a.get("title"),"source":(a.get("source") or {}).get("name"),"publishedAt":a.get("publishedAt"),"url":a.get("url"),"description":a.get("description")})
            else:
                record_failure("gnews", term, r, err or "non-ok response")
        except Exception as e:
            record_failure("gnews", term, None, f"{type(e).__name__}: {e}")

    if newsdata_key:
        try:
            r = requests.get("https://newsdata.io/api/1/news",
                             params={"apikey":newsdata_key,"q":term,"language":"en","country":"gb"}, timeout=30)
            payload, err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("results", [])[:10]:
                    articles.append({"provider":"newsdataio","query":term,"title":a.get("title"),"source":a.get("source_id"),"publishedAt":a.get("pubDate"),"url":a.get("link"),"description":a.get("description")})
            else:
                record_failure("newsdataio", term, r, err or "non-ok response")
        except Exception as e:
            record_failure("newsdataio", term, None, f"{type(e).__name__}: {e}")

    if not newsapi_key and not gnews_key and not newsdata_key:
        try:
            r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                             params={"query":term,"mode":"ArtList","format":"json","maxrecords":10}, timeout=30)
            payload, err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("articles", []):
                    articles.append({"provider":"gdelt","query":term,"title":a.get("title"),"source":a.get("sourceCountry"),"publishedAt":a.get("seendate"),"url":a.get("url"),"description":a.get("summary")})
            else:
                record_failure("gdelt", term, r, err or "non-ok response")
        except Exception as e:
            record_failure("gdelt", term, None, f"{type(e).__name__}: {e}")

# dedupe
seen = set(); deduped = []
for a in articles:
    key = a.get("url") or f"{a.get('title')}|{a.get('publishedAt')}"
    if key in seen: continue
    seen.add(key); deduped.append(a)

news_path = out / "news_articles.json"
fails_path = out / "news_provider_failures.csv"
write_json(news_path, deduped)
pd.DataFrame(failures).to_csv(fails_path, index=False)

mf = manifest_base(step)
add_artifact(mf, news_path, {"count": len(deduped)})
add_artifact(mf, fails_path, {"count": len(failures)})
if len(deduped) == 0:
    mf["status"] = "warning"
    mf["notes"].append("No articles collected; downstream steps should treat news as unavailable rather than fatal.")
write_json(out / "manifest.json", mf)
print("Articles:", len(deduped), "Failures:", len(failures))